CAMADA BRONZE - Entrada e Leitura do csv


In [29]:
import pandas as pd
from datetime import datetime

source = r"C:\Users\pedro.malbuquerque\Downloads\electronics_sales_raw(1).csv"

df = pd.read_csv(
    source
    )

df['filename'] = source
df['ingestion_timestamp'] = datetime.now()


df_bronze = df.copy()

df_bronze.head()

df_bronze.to_csv(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\layer_bronze.csv", index=False)


CAMADA SILVER - PADRONIZAÇÃO DE COLUNAS DE DATA


In [ ]:
df_silver = df_bronze.copy()

df_silver.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             7000 non-null   int64         
 1   customer_id          7000 non-null   object        
 2   customer_name        7000 non-null   object        
 3   customer_segment     7000 non-null   object        
 4   customer_type        7000 non-null   object        
 5   first_purchase_date  7000 non-null   object        
 6   last_purchase_date   7000 non-null   object        
 7   product_id           7000 non-null   object        
 8   product_name         7000 non-null   object        
 9   category             7000 non-null   object        
 10  sub_category         7000 non-null   object        
 11  brand                7000 non-null   object        
 12  order_date           7000 non-null   object        
 13  quantity             7000 non-nul

In [54]:
print(df_silver.columns)
print(df_silver.unit_price.dtype)
print(df_silver.quantity.dtype)
print(df_silver.discount_pct.dtype)
print(df_silver.operating_expenses.dtype)
print(df_silver.cash_balance.dtype)
print(df_silver.debt_balance.dtype)
print(df_silver.monthly_burn.dtype)
print(df_silver.quantity.dtype)

Index(['order_id', 'customer_id', 'customer_name', 'customer_segment',
       'customer_type', 'first_purchase_date', 'last_purchase_date',
       'product_id', 'product_name', 'category', 'sub_category', 'brand',
       'order_date', 'quantity', 'unit_price', 'discount_pct', 'sales_channel',
       'payment_method', 'sales_rep', 'region', 'operating_expenses',
       'cash_balance', 'debt_balance', 'monthly_burn', 'churn_flag',
       'filename', 'ingestion_timestamp'],
      dtype='object')
float64
int64
float64
float64
float64
float64
float64
int64


In [50]:
df_silver.first_purchase_date = pd.to_datetime(df_silver.first_purchase_date, errors='coerce')
df_silver.last_purchase_date = pd.to_datetime(df_silver.last_purchase_date, errors='coerce')
df_silver.order_date = pd.to_datetime(df_silver.order_date, errors='coerce')

df_silver.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             7000 non-null   int64         
 1   customer_id          7000 non-null   object        
 2   customer_name        7000 non-null   object        
 3   customer_segment     7000 non-null   object        
 4   customer_type        7000 non-null   object        
 5   first_purchase_date  7000 non-null   datetime64[ns]
 6   last_purchase_date   7000 non-null   datetime64[ns]
 7   product_id           7000 non-null   object        
 8   product_name         7000 non-null   object        
 9   category             7000 non-null   object        
 10  sub_category         7000 non-null   object        
 11  brand                7000 non-null   object        
 12  order_date           7000 non-null   datetime64[ns]
 13  quantity             7000 non-nul

In [52]:
df_silver.isna().sum()

order_id               0
customer_id            0
customer_name          0
customer_segment       0
customer_type          0
first_purchase_date    0
last_purchase_date     0
product_id             0
product_name           0
category               0
sub_category           0
brand                  0
order_date             0
quantity               0
unit_price             0
discount_pct           0
sales_channel          0
payment_method         0
sales_rep              0
region                 0
operating_expenses     0
cash_balance           0
debt_balance           0
monthly_burn           0
churn_flag             0
filename               0
ingestion_timestamp    0
dtype: int64

In [82]:
df_silver.head(3)

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,payment_method,sales_rep,region,operating_expenses,cash_balance,debt_balance,monthly_burn,churn_flag,filename,ingestion_timestamp
0,10001,C1689,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,MX Keys S,Electronics,...,Credit Card,Eduardo,South,239.83,14629.23,3092.03,329.57,0,C:\Users\pedro.malbuquerque\Downloads\electron...,2026-05-05 10:54:49.609819
1,10002,C1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,Credit Card,Diego,North,81.55,8719.54,1189.63,88.62,1,C:\Users\pedro.malbuquerque\Downloads\electron...,2026-05-05 10:54:49.609819
2,10003,C1422,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,iPad mini,Electronics,...,Cash,Camila,South,244.79,9475.50,3302.27,389.89,0,C:\Users\pedro.malbuquerque\Downloads\electron...,2026-05-05 10:54:49.609819


In [58]:
# 3. Validação de desconto
print("\n🔎 DESCONTOS INVÁLIDOS:")
print("Desconto menor que 0:", (df_silver['discount_pct'] < 0).sum())
print("Desconto maior que 1:", (df_silver['discount_pct'] > 1).sum())

# 4. Validação de preço e quantidade
print("\n🔎 PREÇO E QUANTIDADE INVÁLIDOS:")
print("Preço <= 0:", (df_silver['unit_price'] <= 0).sum())
print("Quantidade <= 0:", (df_silver['quantity'] <= 0).sum())

# 5. Validação temporal
print("\n🔎 DATAS INCONSISTENTES:")
print(
    "Pedido antes da primeira compra:",
    (df_silver['order_date'] < df_silver['first_purchase_date']).sum()
)

print(
    "Última compra antes da primeira compra:",
    (df_silver['last_purchase_date'] < df_silver['first_purchase_date']).sum()
)

df_silver.to_csv(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\layer_silver.csv", index=False)


🔎 DESCONTOS INVÁLIDOS:
Desconto menor que 0: 0
Desconto maior que 1: 0

🔎 PREÇO E QUANTIDADE INVÁLIDOS:
Preço <= 0: 0
Quantidade <= 0: 0

🔎 DATAS INCONSISTENTES:
Pedido antes da primeira compra: 0
Última compra antes da primeira compra: 0


CAMADA GOLD

In [ ]:
df_gold = df_silver.copy()



df_gold['gross_revenue'] = df_gold['unit_price'] * df_gold['quantity']

df_gold['net_revenue'] = df_gold['gross_revenue'] * (1 - df_gold['discount_pct'])


df_gold.to_csv(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\layer_gold.csv", index=False)


CRIAÇÃO DA TABELAS DIM E FACT

In [73]:
dim_customer = df_gold[[
    'customer_id',
    'customer_name',
    'customer_segment',
    'customer_type',
    'first_purchase_date',
    'last_purchase_date',
    'churn_flag'
]].drop_duplicates()

In [72]:
dim_product = df_gold[[
    'product_id',
    'product_name',
    'category',
    'sub_category',
    'brand'
]].drop_duplicates()

In [71]:
dim_calendar = df_gold[['order_date']].drop_duplicates().copy()

dim_calendar['year'] = dim_calendar['order_date'].dt.year
dim_calendar['month'] = dim_calendar['order_date'].dt.month
dim_calendar['day'] = dim_calendar['order_date'].dt.day
dim_calendar['quarter'] = dim_calendar['order_date'].dt.quarter

In [78]:
dim_channel = df_gold[['sales_channel']].drop_duplicates().reset_index(drop=True)
dim_channel['channel_id'] = dim_channel.index + 1

In [79]:
dim_sales = df_gold[['sales_rep', 'region']].drop_duplicates().reset_index(drop=True)
dim_sales['sales_id'] = dim_sales.index + 1

In [80]:
dim_payment = df_gold[['payment_method']].drop_duplicates().reset_index(drop=True)
dim_payment['payment_id'] = dim_payment.index + 1

In [83]:
fact_sales = (
    df_gold
    .merge(dim_channel, on='sales_channel', how='left')
    .merge(dim_payment, on='payment_method', how='left')
    .merge(dim_sales, on=['sales_rep', 'region'], how='left')
)

fact_sales = fact_sales[[
    'order_id',
    'customer_id',
    'product_id',
    'order_date',
    'channel_id',
    'payment_id',
    'sales_id',
    'quantity',
    'unit_price',
    'discount_pct',
    'gross_revenue',
    'net_revenue'
]].copy()

fact_sales = fact_sales.rename(columns={
    'order_date': 'date'
})

fact_sales.head()

,order_id,customer_id,product_id,date,channel_id,payment_id,sales_id,quantity,unit_price,discount_pct,gross_revenue,net_revenue
0,10001,C1689,P1001,2024-04-24,1,1,1,4,109.77,0.05,439.08,417.1260
1,10002,C1002,P1002,2025-07-15,1,1,2,3,350.29,0.15,1050.87,893.2395
2,10003,C1422,P1003,2024-01-01,1,2,3,2,643.98,0.15,1287.96,1094.7660
3,10004,C1308,P1004,2024-01-17,2,1,4,1,844.50,0.00,844.50,844.5000
4,10005,C0826,P1002,2024-12-17,1,1,5,1,332.72,0.05,332.72,316.0840


EXPORTAÇÂO PARA CONSUMO NO BI

In [84]:
fact_sales.to_parquet(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\fact_sales.parquet", index=False)
dim_customer.to_parquet(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\dim_customer.parquet", index=False)
dim_product.to_parquet(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\dim_product.parquet", index=False)
dim_calendar.to_parquet(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\dim_calendar.parquet", index=False)
dim_channel.to_parquet(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\dim_channel.parquet", index=False)
dim_sales.to_parquet(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\dim_sales.parquet", index=False)
dim_payment.to_parquet(r"C:\Users\pedro.malbuquerque\Desktop\CASE ELET\dim_payment.parquet", index=False)